# NB 6.2 &mdash; El laboratori

**MP 5149** &mdash; Desenvolupament de components software per a sistemes d'aprenentatge automàtic

**UT6** &mdash; Xarxes neuronals convolucionals i introducció a la visió per computador

---
### D'on venim

Al NB 6.1 vas descobrir què fa una CNN amb dígits. Però els dígits són un problema fàcil:
blancs sobre negre, centrats i sempre iguals.

### Què farem avui

Avui **el laboratori és teu**. Tens a mà quatre conjunts de dades i una xarxa que es
configura amb paràmetres. Hi faràs una competició amb la resta de la classe, et mesuraràs
amb una CNN en un qüestionari, provocaràs l'overfitting per aprendre a curar-lo i
acabaràs entrenant una xarxa que endevina el que dibuixes.

### Preguntes que has de saber respondre en acabar

1. Quin patró segueixen les capes d'una CNN, i més capes sempre vol dir millor?
2. Quan tria bé Adam, i quan tria bé SGD?
3. Què canvia quan les imatges són en color, i què fa `batch_norm`?
4. Com es reconeix l'overfitting a les corbes, i què fan el dropout i la data augmentation?
5. Què respon una xarxa quan li ensenyes una cosa que no ha vist mai?

## 0. Preparació

Les biblioteques i la mateixa plantilla d'entrenament del NB 6.1.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.neural_network import MLPClassifier
from torchvision.transforms import v2

plt.rcParams["figure.figsize"] = (6, 4)

La **plantilla d'entrenament**, igual que al NB 6.1. No cal modificar-la.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Plantilla d'entrenament. No cal modificar-la: només cal saber què fa cada funció.
# Fa el mateix que MLPClassifier feia per dins a la UT5, però a la vista.
# ─────────────────────────────────────────────────────────────────────────────
import math
import time

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader

SEED = 42
LOSS_LIMIT = 100       # en un entrenament sa, la loss no s'hi acosta mai
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # GPU si n'hi ha; si no, CPU


def make_cnn(in_channels, n_classes, image_size=28, filters=(16, 32), pooling=True,
             global_pool=False, batch_norm=False, dropout=0.0):
    """Crea una CNN a partir de paràmetres, com MLPClassifier amb hidden_layer_sizes.

    Paràmetres:
        in_channels: canals de la imatge (1 en gris, 3 en color).
        n_classes: quantes classes ha de distingir.
        image_size: costat de la imatge en píxels.
        filters: filtres de cada capa convolucional; (16, 32) són dues capes.
        pooling: si cada capa redueix la imatge a la meitat amb MaxPool2d.
        global_pool: si al final es fa la mitjana de cada mapa en lloc d'aplanar-los.
        batch_norm: si s'afegeix BatchNorm2d després de cada convolució.
        dropout: fracció de neurones que s'apaguen a l'atzar a l'entrenament.

    Retorna el model. Exemple: model = make_cnn(1, 10, filters=(16, 32, 64))
    """
    if isinstance(filters, int):
        raise ValueError(f"filters ha de ser una llista de capes: per a una sola capa, escriu filters=({filters},), amb la coma.")
    if not len(filters) or min(filters) < 1:
        raise ValueError("filters necessita almenys una capa, amb un nombre positiu de filtres: per exemple, filters=(16, 32).")
    if not 0 <= dropout < 1:
        raise ValueError(f"dropout és la fracció de neurones que s'apaguen i ha d'anar de 0 a menys d'1: {dropout} no val. Prova dropout=0.5.")
    n_pools = (len(filters) - 1 if global_pool else len(filters)) if pooling else 0
    side = image_size // 2 ** n_pools
    if side < 1:
        raise ValueError(f"Hi ha massa capes amb pooling per a una imatge de {image_size} píxels: "
                         "fes filters més curt o prova pooling=False.")

    torch.manual_seed(SEED)
    layers = []
    channels = in_channels
    for position, n_filters in enumerate(filters):
        layers += [nn.Conv2d(channels, n_filters, kernel_size=3, padding=1)]
        if batch_norm:
            layers += [nn.BatchNorm2d(n_filters)]
        layers += [nn.ReLU()]
        is_last = position == len(filters) - 1
        if pooling and not (global_pool and is_last):
            layers += [nn.MaxPool2d(2)]
        channels = n_filters

    switch_off = [nn.Dropout(dropout)] if dropout > 0 else []
    if global_pool:
        layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), *switch_off, nn.Linear(channels, n_classes)]
    else:
        layers += [nn.Flatten(), nn.Linear(channels * side * side, 64), nn.ReLU(),
                   *switch_off, nn.Linear(64, n_classes)]
    return nn.Sequential(*layers)


def count_parameters(model):
    """Retorna quants pesos entrenables té el model."""
    return sum(weights.numel() for weights in model.parameters() if weights.requires_grad)


def make_loaders(train_set, val_set, batch_size=64, augment=None):
    """Prepara els mini-batches d'entrenament i de validació.

    Paràmetres:
        train_set, val_set: conjunts de dades de PyTorch.
        batch_size: imatges per mini-batch.
        augment: transformació per a cada imatge d'entrenament (data augmentation) o None.

    Retorna (train_loader, val_loader). Només l'entrenament es barreja i s'augmenta.
    """
    def collate_augmented(batch):
        images = torch.stack([augment(image) for image, _ in batch])
        labels = torch.as_tensor([label for _, label in batch])
        return images, labels

    collate = collate_augmented if augment is not None else None
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=collate)
    val_loader = DataLoader(val_set, batch_size=1000)
    return train_loader, val_loader


@torch.no_grad()
def _evaluate(model, loader):
    """Retorna la loss mitjana i l'encert del model amb un loader."""
    model.eval()
    device = next(model.parameters()).device
    loss_function = nn.CrossEntropyLoss(reduction="sum")
    total_loss, hits, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        scores = model(images)
        total_loss += loss_function(scores, labels).item()
        hits += (scores.argmax(dim=1) == labels).sum().item()
        total += len(labels)
    return total_loss / total, hits / total


def accuracy(model, loader):
    """Retorna l'encert del model (de 0 a 1), com score() a scikit-learn."""
    return _evaluate(model, loader)[1]


def train_model(model, train_loader, val_loader, epochs, lr, optimizer="adam"):
    """Entrena el model i mostra com va, epoch a epoch. És el fit() d'aquesta unitat.

    Paràmetres:
        model: la xarxa, feta amb make_cnn.
        train_loader, val_loader: els mini-batches, fets amb make_loaders.
        epochs: quantes vegades es recorren totes les dades d'entrenament.
        lr: el learning rate.
        optimizer: "adam" o "sgd".

    Retorna un diccionari amb les llistes train_loss, val_loss, train_acc i val_acc.
    """
    optimizers = {"adam": torch.optim.Adam, "sgd": torch.optim.SGD}
    if optimizer not in optimizers:
        raise ValueError(f"Optimitzador desconegut: {optimizer!r}. Tria 'adam' o 'sgd'.")

    torch.manual_seed(SEED)
    check_data(model, val_loader)
    model.to(DEVICE)
    step_maker = optimizers[optimizer](model.parameters(), lr=lr)
    loss_function = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        start = time.perf_counter()
        model.train()                                  # activa dropout i batch norm
        total_loss, hits, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            step_maker.zero_grad()                     # esborra el repartiment anterior
            scores = model(images)                     # 1. la xarxa prediu
            _check_labels(scores, labels)
            loss = loss_function(scores, labels)       # 2. mesurem l'error (la loss)
            loss.backward()                            # 3. backpropagation: repartim la culpa
            step_maker.step()                          # 4. una passa de gradient descent
            total_loss += loss.item() * len(labels)
            hits += (scores.argmax(dim=1) == labels).sum().item()
            total += len(labels)

        val_loss, val_acc = _evaluate(model, val_loader)
        history["train_loss"].append(total_loss / total)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(hits / total)
        history["val_acc"].append(val_acc)
        print(f"Epoch {epoch}/{epochs} · loss {total_loss / total:.3f} · "
              f"encert a validació {val_acc:.1%} · {time.perf_counter() - start:.0f} s")

        if not math.isfinite(total_loss) or total_loss / total > LOSS_LIMIT:
            print("La loss s'ha disparat i l'entrenament s'atura: "
                  "segurament el learning rate és massa alt.")
            break
    return history


def check_data(model, loader):
    """Prova la xarxa amb una imatge del conjunt i, si no l'accepta, explica què cal canviar."""
    image, _ = loader.dataset[0]
    model.to(DEVICE)
    model.eval()
    try:
        with torch.no_grad():
            model(image.unsqueeze(0).to(DEVICE))
    except RuntimeError as error:
        channels, height, width = image.shape
        raise ValueError(f"La xarxa no accepta aquestes imatges, que tenen {channels} canals i fan {height} × {width} "
                         f"píxels. Crea-la de nou amb make_cnn({channels}, n_classes, image_size={height}), "
                         "amb la resta de paràmetres que ja tenies.") from error


def _check_labels(scores, labels):
    """Si hi ha més classes que sortides, la loss no es pot calcular: ho expliquem en lloc de fallar."""
    if labels.max() >= scores.shape[1]:
        raise ValueError(f"Aquest conjunt té etiquetes fins a la {labels.max().item()}, però la xarxa només té "
                         f"{scores.shape[1]} sortides. Crea-la de nou amb n_classes igual al nombre de classes: len(names).")


def plot_history(history):
    """Dibuixa les corbes de loss i d'encert d'un entrenament."""
    fig, grafics = plt.subplots(1, 2, figsize=(10, 3.5))
    epochs = range(1, len(history["train_loss"]) + 1)
    grafics[0].plot(epochs, history["train_loss"], "o-", label="entrenament")
    grafics[0].plot(epochs, history["val_loss"], "o-", label="validació")
    grafics[0].set_ylabel("loss")
    if not any(math.isfinite(value) for value in history["train_loss"] + history["val_loss"]):
        grafics[0].text(0.5, 0.5, "La loss s'ha disparat:\nprova un learning rate més petit.",
                        transform=grafics[0].transAxes, ha="center", va="center")
    grafics[1].plot(epochs, history["train_acc"], "o-", label="entrenament")
    grafics[1].plot(epochs, history["val_acc"], "o-", label="validació")
    grafics[1].set_ylabel("encert")
    for grafic in grafics:
        grafic.set_xlabel("epoch")
        grafic.legend()
    plt.tight_layout()
    plt.show()

I les eines del laboratori. N'hi ha unes quantes que aniran sortint: `load_dataset` carrega
les dades, `draw_canvas` obre un llenç per dibuixar i la resta serveix per jugar i comparar.
Tampoc cal modificar-les.

In [ ]:
# Llenç per dibuixar a Colab. No cal modificar-la.
import base64
import io

import numpy as np
from IPython.display import HTML, display
from PIL import Image

CANVAS_PAGE = """
<canvas id="llenc" width="{size}" height="{size}"
        style="background:black;border-radius:6px;touch-action:none;cursor:crosshair"></canvas><br>
<button id="fet" style="margin-top:6px;padding:6px 18px">Fet!</button>
<script>
const canvas = document.getElementById("llenc");
const context = canvas.getContext("2d");
context.strokeStyle = "white"; context.lineWidth = {size} / 28 * 1.6;
context.lineCap = "round"; context.lineJoin = "round";
let drawing = false;
const position = e => {{ const r = canvas.getBoundingClientRect(); return [e.clientX - r.left, e.clientY - r.top]; }};
canvas.onpointerdown = e => {{ drawing = true; context.beginPath(); context.moveTo(...position(e)); }};
canvas.onpointermove = e => {{ if (drawing) {{ context.lineTo(...position(e)); context.stroke(); }} }};
window.onpointerup = () => {{ drawing = false; }};
window.waitForDrawing = () => new Promise(resolve => {{
  document.getElementById("fet").onclick = () => resolve(canvas.toDataURL("image/png"));
}});
</script>
"""


def bitmap_from_png(png):
    """Converteix el dibuix en una imatge de 28×28 centrada, com les de Quick Draw.

    Retorna un array uint8, o None si el llenç és buit.
    """
    image = np.array(Image.open(io.BytesIO(png)).convert("RGBA"))
    ink = image[:, :, :3].max(axis=2) * (image[:, :, 3] > 0)
    rows, cols = np.nonzero(ink > 30)
    if len(rows) == 0:
        print("El llenç és buit: dibuixa alguna cosa i torna a executar la cel·la.")
        return None
    cropped = Image.fromarray(ink[rows.min():rows.max() + 1, cols.min():cols.max() + 1].astype(np.uint8))
    scale = 24 / max(cropped.size)
    new_size = (max(1, round(cropped.width * scale)), max(1, round(cropped.height * scale)))
    bitmap = Image.new("L", (28, 28), 0)
    bitmap.paste(cropped.resize(new_size, Image.LANCZOS), ((28 - new_size[0]) // 2, (28 - new_size[1]) // 2))
    return np.array(bitmap, dtype=np.uint8)


def draw_canvas(size=280):
    """Obre un llenç, espera que premis «Fet!» i retorna el dibuix de 28×28.

    Només funciona a Colab. Retorna None fora de Colab o si el llenç és buit.
    """
    try:
        from google.colab import output
    except ImportError:
        print("El llenç només funciona a Colab. Aquí pots provar el model amb una imatge "
              "del conjunt de test, amb predict_top3.")
        return None
    display(HTML(CANVAS_PAGE.format(size=size)))
    data_url = output.eval_js("waitForDrawing()", timeout_sec=600)
    return bitmap_from_png(base64.b64decode(data_url.split(",", 1)[1]))


# Funcions del laboratori: carregar dades, jugar i comparar. No cal modificar-la.
import io
import tempfile
import unicodedata
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import TensorDataset

URL_BASE = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/desenvolupament-components-software/refs/heads/main/"
URL_CIFAR = "https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text/"
CACHE = Path(tempfile.gettempdir()) / "dades_ut6"
CIFAR_NAMES = ["avió", "cotxe", "ocell", "gat", "cérvol", "gos", "granota", "cavall", "vaixell", "camió"]
NPZ_FILES = {"mnist": "UT01-Xarxes_neuronals_supervisades/mnist/mnist.npz",
             "fashion": "UT06-Xarxes_convolucionals/fashion_mnist/fashion_mnist.npz",
             "quickdraw": "UT06-Xarxes_convolucionals/quickdraw/quickdraw_30.npz"}
N_VAL = {"mnist": 10_000, "fashion": 10_000, "cifar10": 10_000, "quickdraw": 15_000}


def _download(url):
    """Baixa un fitxer només la primera vegada i retorna on és."""
    target = CACHE / url.rsplit("/", 1)[1]
    if not target.exists():
        CACHE.mkdir(parents=True, exist_ok=True)
        try:
            with urllib.request.urlopen(url, timeout=300) as response:
                content = response.read()
        except OSError as error:
            raise RuntimeError(f"No s'han pogut baixar les dades de {url}. Comprova la connexió "
                               f"i torna a executar la cel·la. Detall: {error}") from error
        partial = target.with_suffix(".part")
        partial.write_bytes(content)
        partial.rename(target)
    return target


def _read_cifar(part):
    """Llegeix una part de CIFAR-10 com a imatges (N, 32, 32, 3) i etiquetes."""
    table = pd.read_parquet(_download(f"{URL_CIFAR}{part}-00000-of-00001.parquet"))
    images = np.stack([np.array(Image.open(io.BytesIO(cell["bytes"]))) for cell in table["img"]])
    return images, table["label"].to_numpy()


def _read_arrays(name):
    """Retorna imatges (N, C, H, W) en uint8, etiquetes i noms de classe d'un conjunt."""
    if name == "cifar10":
        X_train, y_train = _read_cifar("train")
        X_test, y_test = _read_cifar("test")
        return X_train.transpose(0, 3, 1, 2), y_train, X_test.transpose(0, 3, 1, 2), y_test, CIFAR_NAMES
    with np.load(_download(URL_BASE + NPZ_FILES[name]), allow_pickle=False) as data:
        names = [str(n) for n in data["class_names"]] if "class_names" in data.files else [str(n) for n in range(10)]
        return (data["X_train"].reshape(-1, 1, 28, 28), data["y_train"],
                data["X_test"].reshape(-1, 1, 28, 28), data["y_test"], names)


def _to_dataset(images, labels):
    """Passa imatges en uint8 a un TensorDataset amb valors de 0 a 1."""
    return TensorDataset(torch.tensor(images / 255.0, dtype=torch.float32),
                         torch.tensor(labels.astype(int)))


def load_dataset(name, n_train=None, seed=42, n_val=None):
    """Carrega un dels quatre conjunts de la unitat, ja preparat per a una CNN.

    Paràmetres:
        name: "mnist", "fashion", "cifar10" o "quickdraw".
        n_train: quantes imatges d'entrenament es volen, triades a l'atzar. None, totes.
        seed: la llavor d'aquest sorteig.
        n_val: quantes imatges de validació es volen. None, totes. Si en són menys, va més de pressa.

    Retorna (train_set, val_set, test_set, class_names).
    """
    if name not in N_VAL:
        raise ValueError(f"Conjunt desconegut: {name!r}. Tria'n un d'aquests: {', '.join(N_VAL)}.")
    X_train, y_train, X_test, y_test, names = _read_arrays(name)
    n_fit = len(X_train) - N_VAL[name]
    if n_train is not None and not 1 <= n_train <= n_fit:
        raise ValueError(f"{name} només té {n_fit} imatges d'entrenament: tria un n_train entre 1 i {n_fit}.")
    chosen = np.arange(n_fit) if n_train is None else np.random.default_rng(seed).permutation(n_fit)[:n_train]
    stop = None if n_val is None else n_fit + n_val
    return (_to_dataset(X_train[chosen], y_train[chosen]), _to_dataset(X_train[n_fit:stop], y_train[n_fit:stop]),
            _to_dataset(X_test, y_test), names)


def _as_picture(image):
    """Prepara un tensor (C, H, W) per dibuixar-lo amb imshow."""
    return image.permute(1, 2, 0).squeeze().cpu().numpy()


def show_examples(dataset, class_names, n=10):
    """Dibuixa les primeres imatges d'un conjunt amb el nom de la seva classe."""
    fig, grafics = plt.subplots(1, n, figsize=(1.2 * n, 1.8))
    for number, grafic in enumerate(grafics):
        image, label = dataset[number]
        grafic.imshow(_as_picture(image), cmap="gray")
        grafic.set_title(class_names[int(label)], fontsize=9)
        grafic.axis("off")
    plt.show()


def show_augmented(image, augment, n=8):
    """Dibuixa la imatge original i n versions que en fa la data augmentation."""
    fig, grafics = plt.subplots(1, n + 1, figsize=(1.2 * (n + 1), 1.8))
    grafics[0].imshow(_as_picture(image), cmap="gray")
    grafics[0].set_title("original", fontsize=9)
    for grafic in grafics[1:]:
        grafic.imshow(_as_picture(augment(image)), cmap="gray")
    for grafic in grafics:
        grafic.axis("off")
    plt.show()


@torch.no_grad()
def predict_top3(model, image, class_names):
    """Retorna les tres classes més probables d'una imatge: [(nom, probabilitat), ...]."""
    model.eval()
    device = next(model.parameters()).device
    probabilities = torch.softmax(model(image.unsqueeze(0).to(device))[0], dim=0).cpu()
    values, positions = probabilities.topk(3)
    return [(class_names[position], value) for position, value in zip(positions.tolist(), values.tolist())]


def keep_classes(dataset, class_names, chosen):
    """Es queda amb les classes triades i les renumera des de 0.

    Retorna (conjunt reduït, llista de noms en l'ordre de chosen).
    """
    for name in chosen:
        if name not in class_names:
            raise ValueError(f"La classe {name!r} no hi és. Les que hi ha: {', '.join(class_names)}.")
    images, labels = dataset.tensors
    new_labels = torch.full_like(labels, -1)
    for number, name in enumerate(chosen):
        new_labels[labels == class_names.index(name)] = number
    keep = new_labels >= 0
    return TensorDataset(images[keep], new_labels[keep]), list(chosen)


def pictionary(model, class_names):
    """Obre el llenç, passa el dibuix pel model i mostra les tres classes més probables.

    Retorna el top-3, o None si no hi ha dibuix (llenç buit o fora de Colab).
    """
    drawing = draw_canvas()
    if drawing is None:
        return None
    image = torch.tensor(drawing / 255.0, dtype=torch.float32).unsqueeze(0)
    top = predict_top3(model, image, class_names)
    fig, grafics = plt.subplots(1, 2, figsize=(7, 2.5))
    grafics[0].imshow(drawing, cmap="gray")
    grafics[0].set_title("el que veu la xarxa")
    grafics[0].axis("off")
    grafics[1].barh([name for name, _ in top][::-1], [value for _, value in top][::-1], color="#1C7293")
    grafics[1].set_xlim(0, 1)
    grafics[1].set_xlabel("probabilitat")
    plt.tight_layout()
    plt.show()
    return top


def make_quiz(dataset, n=20, seed=7):
    """Tria sempre les mateixes n imatges d'un conjunt per al qüestionari."""
    chosen = np.random.default_rng(seed).permutation(len(dataset))[:n]
    images, labels = dataset.tensors
    return TensorDataset(images[chosen], labels[chosen])


def show_quiz(quiz, class_names):
    """Dibuixa les imatges del qüestionari numerades, sense dir-ne la classe."""
    print("Classes possibles:", ", ".join(class_names))
    fig, grafics = plt.subplots(4, 5, figsize=(9, 7.5))
    for number, grafic in enumerate(grafics.flat):
        grafic.imshow(_as_picture(quiz[number][0]), cmap="gray", interpolation="nearest")
        grafic.set_title(number + 1)
        grafic.axis("off")
    plt.show()


def _plain(text):
    """Treu majúscules, accents i espais dels extrems per comparar respostes."""
    return "".join(letter for letter in unicodedata.normalize("NFD", text.strip().lower())
                   if not unicodedata.combining(letter))


@torch.no_grad()
def quiz_results(model, quiz, answers, class_names):
    """Compara les respostes d'una persona i les del model amb la classe real.

    Retorna una taula amb les columnes real, persona i CNN, i mostra el marcador.
    """
    images, labels = quiz.tensors
    if len(answers) != len(images):
        raise ValueError(f"Calen {len(images)} respostes i n'hi ha {len(answers)}.")
    known = {_plain(name): name for name in class_names}
    unknown = [answer for answer in answers if _plain(answer) not in known]
    if unknown:
        raise ValueError(f"Resposta desconeguda: {unknown[0]!r}. Les classes són: {', '.join(class_names)}.")
    model.eval()
    device = next(model.parameters()).device
    guesses = model(images.to(device)).argmax(dim=1).cpu().tolist()
    table = pd.DataFrame({"real": [class_names[label] for label in labels.tolist()],
                          "persona": [known[_plain(answer)] for answer in answers],
                          "CNN": [class_names[guess] for guess in guesses]},
                         index=range(1, len(images) + 1))
    print(f"Persona: {(table['persona'] == table['real']).sum()} de {len(table)} · "
          f"CNN: {(table['CNN'] == table['real']).sum()} de {len(table)}")
    return table

## 1. Quatre conjunts de dades, una sola funció

`load_dataset(nom)` et dona qualsevol dels quatre conjunts de la unitat, ja preparats per a
una CNN. Sempre retorna el mateix: entrenament, validació, test i els noms de les classes.

| Nom | Què hi ha | Entrenament | Validació | Forma de cada imatge | Classes |
|---|---|---|---|---|---|
| `mnist` | Dígits escrits a mà | 50.000 | 10.000 | 1 × 28 × 28 | 10 |
| `fashion` | Fotos de peces de roba | 50.000 | 10.000 | 1 × 28 × 28 | 10 |
| `cifar10` | Fotos en color d'animals i vehicles | 40.000 | 10.000 | 3 × 32 × 32 | 10 |
| `quickdraw` | Dibuixos fets a corre-cuita | 60.000 | 15.000 | 1 × 28 × 28 | 30 |

El paràmetre `n_train` tria quantes imatges d'entrenament vols. Amb menys imatges, cada
prova tarda menys. Comencem per **Fashion-MNIST**, el conjunt de l'exercici 8 del NB 1.2,
amb 20.000 imatges.

In [ ]:
train_set, val_set, test_set, names = load_dataset("fashion", n_train=20_000)
train_loader, val_loader = make_loaders(train_set, val_set)

print("Entrenament:", len(train_set), "· Validació:", len(val_set))
print("Forma d'una imatge:", tuple(train_set[0][0].shape))
show_examples(train_set, names)

Per saber si una CNN aporta res, primer ens cal una referència. Aquest és el mateix MLP del
NB 1.2, de 128 neurones, amb les mateixes imatges. `MLPClassifier` les vol en fila, i per això
les aplanem. Limitem-lo a 10 epochs, com vam fer amb MNIST.

In [ ]:
X_flat = train_set.tensors[0].flatten(start_dim=1).numpy()
y_flat = train_set.tensors[1].numpy()
X_val_flat = val_set.tensors[0].flatten(start_dim=1).numpy()
y_val_flat = val_set.tensors[1].numpy()

mlp = MLPClassifier(hidden_layer_sizes=(128,), batch_size=64, learning_rate_init=0.001,
                    max_iter=10, random_state=SEED)
start = time.perf_counter()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")            # avisa que voldria més epochs: ja ho sabem
    mlp.fit(X_flat, y_flat)
print(f"MLP: encert de validació {mlp.score(X_val_flat, y_val_flat):.1%} · {time.perf_counter() - start:.0f} s")

**86,7% d'encert en un parell de segons.** Aquest és el llistó.

## 2. El patró de les capes

Gairebé totes les CNN segueixen el mateix patró, que `make_cnn` fa per tu:

1. **Blocs de filtres i pooling.** Cada bloc fa una convolució, una ReLU i un pooling.
2. **Cada bloc, la imatge més petita i amb més filtres.** La imatge es va reduint, i per no
   perdre informació es guarden més mapes. Així es passa de saber *on* hi ha coses a
   saber *què* hi ha.
3. **Capes finals que decideixen**, les que ja coneixes de l'MLP.

| Després de… | Mida dels mapes amb `filters=(16, 32)` | Mapes | Números en total |
|---|---|---|---|
| La imatge | 28 × 28 | 1 | 784 |
| El bloc 1 | 14 × 14 | 16 | 3.136 |
| El bloc 2 | 7 × 7 | 32 | 1.568 |

### Ara tu · Missió 1: el patró, a prova

1. **Aposta.** Quina d'aquestes configuracions creus que traurà més encert en 3 epochs? Quina
   creus que serà la pitjor? Superaran totes l'MLP? `(8, 16)`, `(16, 32)`, `(32, 64)` i
   `(16, 32, 64, 128)`.
2. **Experimenta.** Per a cadascuna, crea-la amb `make_cnn(1, 10, filters=...)` i entrena-la amb
   `train_model(model, train_loader, val_loader, epochs=3, lr=0.001)`. Mesura'n el temps.
3. **Fes una taula** amb filtres, pesos, temps i encert de validació, i afegeix-hi l'MLP.
4. **Explica.** La xarxa més llarga ha estat la millor? Pensa quina mida té la imatge després
   de quatre poolings i quantes epochs li has donat. La diferència amb l'MLP és tan gran com
   amb MNIST? Quant temps costa cada punt d'encert?

> ⏱ Els quatre entrenaments junts tarden un minut i mig a un ordinador normal.

In [ ]:
# El teu codi aquí

## 3. Adam o SGD

Fins ara, `train_model` ha fet servir Adam. L'alternativa és el gradient descent amb
mini-batches de sempre, que PyTorch anomena **SGD**: tots els pesos fan una passa de la mateixa
longitud relativa, sense adaptar-la. Es tria amb `optimizer="sgd"`.

Recordatori del NB 6.1: Adam adapta la passa de cada pes. Per això funciona bé amb valors de
learning rate tan petits com 0,001. **Això vol dir que els dos es fan servir igual?**

### Ara tu · Missió 2: mateixa xarxa, dues maneres de baixar la muntanya

1. **Aposta.** Si entrenes la mateixa xarxa 5 epochs amb Adam i `lr=0.001`, i amb SGD i el mateix
   `lr=0.001`, qui guanyarà i per quant? I si a SGD li dones `lr=0.1`?
2. **Experimenta.** Entrena `make_cnn(1, 10)` en aquests tres casos. Cal una xarxa nova per a cada
   cas. Per triar-lo: `train_model(..., optimizer="sgd")`.
3. **Dibuixa** les tres corbes de `history["val_loss"]` al mateix gràfic, amb `plt.plot` i `label`.
4. **Explica.** És just dir que «SGD és pitjor» després d'haver-lo provat amb un sol learning
   rate? Quin d'aquests dos optimitzadors triaries si només poguessis fer un intent?

> ⏱ Un minut i mig a un ordinador normal.

In [ ]:
# El teu codi aquí

## 4. El color: CIFAR-10

Ara el salt important: **fotos de veritat**. CIFAR-10 té imatges d'avions, cotxes, ocells,
gats, cérvols, gossos, granotes, cavalls, vaixells i camions, en color i de només 32 × 32 píxels
(Krizhevsky, 2009, *Learning Multiple Layers of Features from Tiny Images*).
La primera vegada, baixar-lo tarda uns segons.

#### Un incís: els canals de color

Una imatge en color es guarda com **tres imatges alhora**: una de vermell, una de verd i una de
blau. Cada píxel són tres números, com ara `(1, 0, 0)` per a un vermell pur i `(1, 1, 0)`
per a un groc. Per això la forma és `(3, 32, 32)`.

**Per a què serveix saber-ho?** Cada filtre de la primera capa té ara una lupa per canal.
En gris, un filtre de 3 × 3 tenia 3 × 3 + 1 = 10 pesos, comptant el biaix. En color, en té
3 × 3 × 3 + 1 = 28. A `make_cnn` només cal indicar-ho amb `in_channels=3` i `image_size=32`.

Per a la competició farem servir 10.000 imatges d'entrenament.

In [ ]:
train_set, val_set, test_set, names = load_dataset("cifar10", n_train=10_000)
train_loader, val_loader = make_loaders(train_set, val_set)

print("Forma d'una imatge:", tuple(train_set[0][0].shape))
show_examples(train_set, names)

La referència: la mateixa xarxa de sempre, adaptada al color.

> ⏱ Uns 20 segons a un ordinador normal, i un minut al CPU de Colab.

In [ ]:
model = make_cnn(in_channels=3, n_classes=10, image_size=32)
print("Pesos:", count_parameters(model))
history = train_model(model, train_loader, val_loader, epochs=5, lr=0.001)
plot_history(history)

**49,2%.** Amb MNIST arribàvem al 98%. Encertant a l'atzar es trauria un 10%, i per tant la
xarxa aprèn, però **aquest problema és molt més difícil**: un gat pot estar de cara o d'esquena,
a dalt o a baix, sobre herba o sobre un sofà. I amb 32 × 32 píxels, ni tu n'estaràs sempre segur.

#### Un incís: `batch_norm`

Dins la xarxa, els mapes d'un filtre poden donar valors entre 0 i 90, i els d'un altre, entre 0 i
0,3. Les capes següents han de treballar amb escales molt diferents, i els valors a més a més
van canviant a mesura que s'entrena. La **batch normalization** corregeix els valors de cada filtre
a cada mini-batch perquè la seva mitjana sigui propera a 0 i la seva dispersió, propera a 1.

| Sortida d'un filtre en un mini-batch | Mitjana | Després de `batch_norm` |
|---|---|---|
| 10, 50, 90 | 50 | −1,2, 0, 1,2 |
| 0,1, 0,2, 0,3 | 0,2 | −1,2, 0, 1,2 |

**Per a què serveix?** Amb tots els filtres a la mateixa escala, cada capa no ha de perseguir
els canvis de l'anterior. A `make_cnn` s'activa amb `batch_norm=True`. **Té premi? Ho hauràs de
descobrir.**

### Ara tu · Missió 3: la competició

**Objectiu:** l'encert de validació més alt de la classe amb CIFAR-10. La persona guanyadora
explica a la pissarra què ha fet.

| Regla | Valor |
|---|---|
| Imatges d'entrenament | Les 10.000 que ja tens carregades |
| Pesos | **Màxim 150.000.** Comprova-ho amb `count_parameters` |
| Epochs | **Màxim 5** |
| Puntuació | `history["val_acc"][-1]`: l'última epoch, perquè és el model que et quedes |
| Què pots tocar | `filters`, `pooling`, `global_pool`, `batch_norm`, `dropout`, `lr`, `batch_size` |

1. **Aposta.** Quin paràmetre creus que ajudarà més? Apunta-ho abans de provar res.
2. **Experimenta.** Prova almenys quatre configuracions. **Canvia'n una sola cosa cada vegada**,
   perquè si no, no sabràs què ha funcionat. Guarda-les en una taula.
3. **Puja a la pissarra** el teu millor resultat i la seva configuració.
4. **Explica.** Què ha funcionat més, i què ha fet nosa? Mira també la corba: el teu millor model
   estava millorant, o ja s'havia aturat? Desa'l amb `best_model = model`: el faràs servir tot
   seguit.

> ⏱ Cada intent tarda entre 20 segons i un minut a un ordinador normal, i fins a tres minuts al
> CPU de Colab. Si en vols fer molts, activa la GPU de Colab (Entorn d'execució → Canvia el tipus
> d'entorn), però amb CPU n'hi ha prou.

In [ ]:
# El teu codi aquí

## 5. Persona contra CNN

Una mica més del 50% d'encert sembla poc. **Ho faries gaire millor?** La funció `make_quiz` tria sempre les
mateixes 20 imatges de validació, i `show_quiz` te les ensenya sense dir-te quina classe tenen.

In [ ]:
quiz = make_quiz(val_set)
show_quiz(quiz, names)

### Ara tu · Missió 4: et mesures amb la xarxa

1. **Aposta.** Quantes de les 20 encertaràs? I la xarxa? En quines imatges creus que fallareu tots dos?
2. **Respon.** Sense preguntar a ningú, fes una llista amb les 20 respostes, en ordre:
   `answers = ["gat", "avió", ...]`. Els accents i les majúscules no importen.
3. **Compara** amb `quiz_results(best_model, quiz, answers, names)`, que mostra el marcador i una
   taula. Si no has desat cap model, fes servir `model`.
4. **Explica.** Quantes vegades heu fallat tots dos en la mateixa imatge? Mira on falla només la
   xarxa: és una errada que tu no faries mai? I on fallaves només tu? Què fa servir la xarxa per
   decidir que tu no fas servir, i a l'inrevés? Recorda que només ha vist 10.000 imatges
   com aquestes, i tu has vist el món.

In [ ]:
# El teu codi aquí

## 6. Provocar l'overfitting

Fins ara, poques epochs i moltes imatges. Ara farem el contrari, a posta: **només 2.000 imatges i
moltes epochs**. A la UT5 vas veure que, si una xarxa entrena massa, comença a memoritzar detalls
de les seves imatges en lloc d'aprendre el que serveix per a les altres. És l'**overfitting**.

Com que farem 40 epochs, validarem amb només 2.000 imatges perquè vagi més de pressa: per això
`n_val`.

> ⏱ Uns 35 segons a un ordinador normal, i menys de dos minuts al CPU de Colab.

In [ ]:
small_train, small_val, _, names = load_dataset("cifar10", n_train=2_000, n_val=2_000)
small_train_loader, small_val_loader = make_loaders(small_train, small_val)

model = make_cnn(3, 10, image_size=32)
history = train_model(model, small_train_loader, small_val_loader, epochs=40, lr=0.001)
plot_history(history)

Mira la **corba de loss**. La d'entrenament no para de baixar. La de validació baixa fins a
**1,47**, s'atura cap a l'epoch 20 i torna a pujar fins a **1,69**. A l'última epoch, l'encert és del
**80,8% a entrenament i del 48,4% a validació**. Aquesta distància és l'overfitting:
la xarxa cada vegada és millor amb les imatges que ja coneix, però no pas amb les noves.

Hi ha dues medicines clàssiques.

#### Un incís: el dropout

El **dropout** apaga a l'atzar una fracció de neurones de les capes finals, a cada mini-batch i
només durant l'entrenament. Amb `dropout=0.5` i 64 neurones, a cada passa en treballen unes 32, i
cada vegada són unes altres.

**Per a què serveix?** Una neurona no es pot refiar d'una altra per memoritzar un detall, perquè
pot ser que aquesta no hi sigui. És com un equip en què cada dia falta la meitat: tothom ha de
saber fer una mica de tot. En validar, totes les neurones tornen a treballar.

#### Un incís: la data augmentation

A l'exercici 7 del NB 1.2 vas fer més gran el conjunt amb còpies desplaçades dels dígits. Era
l'**augment de dades** (*data augmentation*), i tenia un inconvenient: la memòria. Aquí fem el
mateix sense guardar-ne cap còpia: **cada vegada que una imatge entra a la xarxa, es transforma una
mica a l'atzar**.

Amb 2.000 imatges i 40 epochs, la xarxa veu 80.000 imatges, totes una mica diferents. **Per
a què serveix?** Un gat continua sent un gat si el mires en un mirall o una mica més a la dreta,
i així la xarxa no pot memoritzar on és cada píxel. Es fa amb `torchvision.transforms.v2`:

| Transformació | Què fa |
|---|---|
| `v2.RandomHorizontalFlip()` | La meitat de les vegades, la gira com un mirall |
| `v2.RandomCrop(32, padding=4)` | La desplaça fins a 4 píxels en qualsevol direcció |
| `v2.Compose([a, b])` | Aplica les dues, una darrere l'altra |

In [ ]:
flip_and_crop = v2.Compose([v2.RandomHorizontalFlip(), v2.RandomCrop(32, padding=4)])
show_augmented(small_train[1][0], flip_and_crop)

Per fer-la servir, `make_loaders` la rep com a paràmetre, i només s'aplica a l'entrenament:

```python
augmented_loader, small_val_loader = make_loaders(small_train, small_val, augment=flip_and_crop)
```

Compte: no totes les transformacions serveixen per a tot. Un 6 girat és un 9.

### Ara tu · Missió 5: cura-la

1. **Aposta.** Tens tres tractaments: `dropout=0.5`, només el mirall i el mirall més el
   desplaçament. Quin creus que tancarà més la distància entre entrenament i validació? Quin creus
   que traurà més encert de validació? Són el mateix?
2. **Experimenta.** Entrena 40 epochs cada tractament, sempre amb una xarxa nova.
3. **Fes una taula** amb el model sense tractar i els tres tractaments: encert d'entrenament i de
   validació de l'última epoch i la loss de validació mínima, `min(history["val_loss"])`.
4. **Explica.** Mira les corbes. Han curat l'overfitting? Han fet miracles amb l'encert?
   Quin problema no poden resoldre cap de les dues medicines? Al NB 6.4 en veuràs una altra
   solució.

> ⏱ Cada entrenament, uns 35 segons a un ordinador normal. En total, menys de dos minuts.

In [ ]:
# El teu codi aquí

## 7. Pictionary

Ara, el moment de jugar. **Quick Draw** és un joc de Google en què la gent dibuixa una cosa en 20
segons. Tenim 30 classes dels seus dibuixos, de 28 × 28 píxels: 2.000 d'entrenament per classe.

In [ ]:
draw_train, draw_val, _, draw_names = load_dataset("quickdraw")
print(", ".join(draw_names))
show_examples(draw_train, draw_names)

Com que no totes les classes són igual de divertides, en triaràs unes quantes. `keep_classes`
s'hi queda i les torna a numerar des de 0.

La funció `pictionary(model, class_names)` obre un llenç: dibuixa amb el ratolí o el dit, prem
**Fet!** i veuràs les tres classes més probables segons la xarxa. És la **softmax** del NB 1.2:
la xarxa no diu només una resposta, sinó quina probabilitat dona a cada classe. El dibuix es
retalla i es centra, com els de Quick Draw.

> El llenç només funciona a Colab. Fora de Colab, no hi ha dibuix ni predicció i la funció t'ho
> explica. Hi pots provar el model amb un dibuix de validació:
> `predict_top3(model, my_val[0][0], my_names)`.

### Ara tu · Missió 6: la xarxa que endevina dibuixos

1. **Tria** entre 5 i 10 classes, amb un nom exactament com els de la llista:
   `chosen = ["gat", "pizza", ...]`. **Aposta:** quines dues es confondran més?
2. **Prepara-les i entrena.** `my_train, my_names = keep_classes(draw_train, draw_names, chosen)`.
   Fes el mateix amb `draw_val`. Crea els loaders, una xarxa amb `make_cnn(1, len(chosen))`
   i entrena 5 epochs.
3. **Juga.** Executa `pictionary(model, my_names)` tantes vegades com vulguis. Fes-ho per parelles:
   qui aconsegueix que la xarxa digui «gat» amb menys traços?
4. **Enganya-la.** Dibuixa una cosa que **no** sigui a les teves classes, com una cara o una
   lletra. Què diu? Amb quina probabilitat? Què t'indica això sobre la probabilitat que dona una
   xarxa?
5. **Explica.** Quines classes confon? Són les que havies apostat? Hi ha alguna classe en què
   hagi de ser fàcil encertar perquè tothom la dibuixa igual?

> ⏱ L'entrenament tarda uns 20 segons a un ordinador normal. El llenç espera fins que premis
> **Fet!**: per això no l'hem posat a cap cel·la del notebook que s'executi sola.

In [ ]:
# El teu codi aquí
# Una cel·la per entrenar i una altra per jugar: així no reentrenes a cada dibuix.

## 8. Resum

| Què hem vist | Què hem après |
|---|---|
| **El patró de capes** | La imatge es redueix i els filtres creixen. Més capes no és automàticament millor |
| **Adam i SGD** | Adam és bo amb poca feina. SGD pot acostar-s'hi, però cal buscar-li el learning rate |
| **Color** | Tres canals, i un problema molt més difícil que els dígits |
| **`batch_norm`** | Posa cada filtre a la mateixa escala |
| **Overfitting** | Es veu quan la loss de validació torna a pujar |
| **Dropout i data augmentation** | Redueixen l'overfitting, però no substitueixen tenir més dades |
| **Softmax** | Sempre dona una resposta, encara que no en tingui cap de bona |

## Exercicis finals

Les sis missions les has fet pel camí. Aquests dos exercicis et demanen tot el notebook.

**Exercici 7. Camp de proves.** Tria un dels quatre conjunts de `load_dataset` i busca-hi la
millor configuració que puguis, amb tot el que has après avui, i amb un màxim de 2 minuts per
entrenament. Porta a classe **la sorpresa més gran que hagis trobat**: una cosa que esperaves
que funcionés i no ho ha fet, o a l'inrevés. Explica-la amb una taula o un gràfic.

**Exercici 8. Obrim el test.** Mesura el model guanyador de la teva competició amb el test:
`load_dataset("cifar10")` el torna com a tercer valor, i `accuracy(model, loader)` en
calcula l'encert. S'assembla a la validació? Si tota la classe tria el millor model mirant la
validació, per què és raonable sospitar que el guanyador hagi tingut una mica de sort?

## Per al debat de classe

1. **La competició.** Si tornéssim a entrenar la xarxa guanyadora amb una altra llavor, tornaria
   a guanyar? Com ho comprovaries? Quines diferències de la pissarra són reals i quines són sort?
2. **Persona contra CNN.** Si la xarxa encerta menys que tu amb fotos de 32 píxels, per què
   els mòbils reconeixen tan bé el que fotografies? Què els falta aquí?
3. **La probabilitat del Pictionary.** Si una xarxa d'un hospital diu «99%» davant d'una imatge
   d'un tipus que no ha vist mai, qui se n'ha d'adonar?